# BO Forge constrained mixed-variable campaign

This notebook demonstrates the v0.4.2 `CampaignSession` workflow for a constrained mixed-variable single-objective campaign.

The public CSV log still stores user-facing values and category labels. Constraints are checked during validation and suggestions are filtered before they are returned.

## 1. Setup

The example uses a mixed campaign with two feasibility constraints and an encoded-space near-duplicate threshold.

In [ ]:
from dataclasses import replace
from pathlib import Path
import os
import shutil
import sys

import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

mpl_cache = PROJECT_ROOT / ".matplotlib-cache"
mpl_cache.mkdir(exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(mpl_cache))

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from bo_forge.errors import SuggestionError
from bo_forge.session import CampaignSession

In [ ]:
config_path = PROJECT_ROOT / "configs" / "06_mixed_constrained_logei.yaml"
seed_log_path = PROJECT_ROOT / "examples" / "06_mixed_constrained_logei_campaign_log.csv"
working_log_path = PROJECT_ROOT / "examples" / "06_mixed_constrained_logei_working_log.csv"
latest_suggestion_path = PROJECT_ROOT / "examples" / "06_mixed_constrained_logei_latest_suggestions.csv"
report_path = PROJECT_ROOT / "reports" / "06_constrained_mixed_campaign_report.txt"
progress_path = PROJECT_ROOT / "reports" / "06_constrained_mixed_progress.pdf"
diagnostics_path = PROJECT_ROOT / "reports" / "06_constrained_mixed_diagnostics.pdf"
TARGET_OBSERVED_ROWS = 15

shutil.copyfile(seed_log_path, working_log_path)

campaign = CampaignSession.from_files(config_path=config_path, log_path=working_log_path)

## 2. Inspect campaign state and constraints

In [ ]:
campaign.validate()

constraints = pd.DataFrame(
    [
        {"name": constraint.name, "expression": constraint.expression}
        for constraint in campaign.config.constraints
    ]
)

display(constraints)
display(campaign.summary())
display(campaign.next_action())
campaign.df

## 3. Generate feasible initial suggestions

The seed log is below `initial_design_size`, so these are Sobol initial-design suggestions filtered through the configured constraints.

In [ ]:
initial_suggestions = campaign.suggest_next(batch_size=2)
initial_suggestions.to_csv(latest_suggestion_path, index=False)

display(initial_suggestions)
display(campaign.suggestion_quality(initial_suggestions))

## 4. Synthetic constrained objective

The simulator mirrors the config constraints: water is allowed only for longer reaction times and low base equivalents.

In [ ]:
def simulate_yield(row):
    loading = float(row["catalyst_loading"])
    reaction_time = int(row["reaction_time"])
    time_scaled = (reaction_time - 10) / 50
    base = float(row["base_equivalents"])
    solvent = row["solvent"]

    if solvent == "Water" and base >= 0.5:
        raise ValueError("Water is infeasible with high base equivalents.")
    if solvent == "Water" and reaction_time < 35:
        raise ValueError("Water is feasible only for longer reaction times.")

    solvent_bonus = {"MeCN": 5.0, "EtOH": 2.0, "Water": 0.5}[solvent]
    base_bonus = {0.1: -4.0, 0.2: 2.5, 0.5: 5.0, 1.0: 1.0}[base]
    peak = np.exp(
        -0.5
        * (
            ((loading - 0.12) / 0.035) ** 2
            + ((time_scaled - 0.55) / 0.22) ** 2
        )
    )
    interaction = 3.0 * np.sin(10.0 * loading + 2.0 * time_scaled)
    return float(45.0 + 25.0 * peak + interaction + base_bonus + solvent_bonus)

## 5. Record the initial suggestions

In [ ]:
campaign.append_suggestions(initial_suggestions)

for _, suggestion in initial_suggestions.iterrows():
    campaign.mark_observed(
        row_id=str(suggestion["row_id"]),
        objective_value=simulate_yield(suggestion),
    )

display(campaign.summary())
campaign.df.tail(4)

## 6. Request a constrained qLogEI batch

The BO batch is decoded, repaired, checked against exact duplicates, checked against the encoded-space distance threshold, and filtered by constraints before it is returned.

This is a practical repair/filter/retry workflow, not a probabilistic constrained acquisition with a learned feasibility model. For qLogEI batches, the `acquisition` value shown on each row is the shared batch-level value.

In [ ]:
bo_suggestions = campaign.suggest_next(batch_size=2)
bo_suggestions.to_csv(latest_suggestion_path, index=False)

display(bo_suggestions)
display(campaign.suggestion_quality(bo_suggestions))

In [ ]:
campaign.append_suggestions(bo_suggestions)

for _, suggestion in bo_suggestions.iterrows():
    campaign.mark_observed(
        row_id=str(suggestion["row_id"]),
        objective_value=simulate_yield(suggestion),
    )

display(campaign.summary())
campaign.df.tail(4)

## 7. Reports and diagnostics

The report and plots use the same session state. Constraint and distance checks affect suggestions, not the canonical CSV schema.

In [ ]:
target_records = []
target_quality = []
relaxed_distance_for_depth = False

while len(campaign.observed_data()) < TARGET_OBSERVED_ROWS:
    remaining = TARGET_OBSERVED_ROWS - len(campaign.observed_data())
    batch_size = min(campaign.config.bo.batch_size, remaining)
    try:
        suggestions = campaign.suggest_next(batch_size=batch_size)
    except SuggestionError as error:
        if "min_normalized_distance" not in str(error):
            raise
        campaign.config = replace(
            campaign.config,
            bo=replace(campaign.config.bo, min_normalized_distance=0.0),
        )
        relaxed_distance_for_depth = True
        suggestions = campaign.suggest_next(batch_size=batch_size)

    suggestions.to_csv(latest_suggestion_path, index=False)
    target_quality.append(campaign.suggestion_quality(suggestions))
    campaign.append_suggestions(suggestions)

    for _, suggestion in suggestions.iterrows():
        value = simulate_yield(suggestion)
        campaign.mark_observed(str(suggestion["row_id"]), value)
        target_records.append(
            {
                "source": suggestion["source"],
                "catalyst_loading": float(suggestion["catalyst_loading"]),
                "reaction_time": int(suggestion["reaction_time"]),
                "base_equivalents": float(suggestion["base_equivalents"]),
                "solvent": suggestion["solvent"],
                "yield_score": value,
            }
        )
    campaign.reload()

assert len(campaign.observed_data()) == TARGET_OBSERVED_ROWS

if relaxed_distance_for_depth:
    print("Relaxed min_normalized_distance to finish the dense 15-step simulated demo.")
if target_records:
    display(pd.DataFrame(target_records))
if target_quality:
    display(pd.concat(target_quality, ignore_index=True).tail(6))

display(campaign.summary())
display(campaign.next_action())

In [ ]:
report = campaign.report()
campaign.export_report(report_path)
display(report["summary"])
display(report["best_observation"])

In [ ]:
campaign.plot_progress(save_path=progress_path);

In [ ]:
campaign.plot_diagnostics(save_path=diagnostics_path);